## Data Wrangling

Recall our data wrangling process.  

1. Discovering
2. Structuring
3. Cleaning 
4. Enriching
5. Validating
6. Publishing

We explored this a bit with Part 1. Let's do a bit more today with Part 2 so we can look at a few more techniques. Mainly we'll look at:
* categories and groupby for discovering
* feature scaling and extraction for structuring
* dirty data and outliers for cleaning

This will be followed by our next lab. 

### 1. Discovering

We are going to continue using the data concerning college student food preferences. This data is the result of a student survey at Mercyhurst University. The file is: food_coded.csv

Data is from https://www.kaggle.com/datasets/borapajo/food-choices/data. About the data: "This dataset includes information on food choices, nutrition, preferences, childhood favorites, and other information from college students. There are 126 responses from students. Data is raw and uncleaned."

Let's do our import and load this data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# needed for structuring step - scaling
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Load the dataset
df = pd.read_csv("food_coded.csv")

In [ ]:
# Do our typical review of the data to get started
# see first five and last five observations

# Note that this does not show us all columns
pd.pandas.set_option('display.max_columns', None)

df

In [ ]:
df.describe()

In [ ]:
df.info()

Let's use a bit of grouping to learn about the data. However, we have quite a few fields that should be categories (factors) instead of numbers or objects.

Let's consider the features: gender, coffee, and cuisine.

In [ ]:
# convert to factors/categories
df["Gender"] = df["Gender"].astype("category")
df["coffee"] = df["coffee"].astype("category")
df["cuisine"] = df["cuisine"].astype("category")

In [ ]:
# explore what we have with groupby
# common aggregation functions are sum, mean, size, count, min, max, std, var, first, and last
# recall that as_index=False gets us a dataframe as the result
gender_group = df.groupby("Gender", as_index=False).size()
gender_group



In [ ]:
# don't forget that we can groupby multiple
gender_group = df.groupby(["Gender", "coffee"], as_index=False).size()
gender_group

### 2. Structuring

Next we'll structure the data. We want to standardize the format and fix any formatting issues.  Are the units okay? Do we have any date/time issues? Are there case issues? ...

Last time we explored the following functions:
* value_counts() for features that are categorical, we can see the unique categories and their counts
* astype() for converting column to correct data type
* fillna() for replacing NaNs with a number
* string manipulation functions (e.g. replace(), strip(), lower(), ...)
* and more

Let's also look at feature scaling which may be necessary. Again, the choice of whether or not to scale a feature depends on situation.  

And then we'll look at how to extract features from an existing feature.  

Example 1: Let's say we are going to to use k-means clustering to place observations into clusters. K-means requires feature scaling.  So let's use normalization, often called min-max scaling to appropriately convert the feature. This will make the minimum value of the feature 0 and the max value 1, and scale values in-between this range appropriately. While we would need to do this for many of the features, let's for now just do this for calories_scone. Note: we **can** do this for multiple features at the same time. 

Steps are:
1. Create scaler
2. Apply scaler
3. Update/Get DataFrame 

We'll use the MinMaxScaler() function. The default range is between 0 and 1, but we can customize that with the function. This is part of the preprocessing module of the sklearn library. We'll be using this library more in the future. 

Note that your book does this all in one line of code:
<code>normalizedArray = preprocessing.MinMaxScaler().fit_transform(original)</code>


In [ ]:
# Let's see what we have first
df["calories_scone"].describe()

Let's make sure the data is ready for normalization first. Currently the data is a float, so our data type is okay. But we can check for nulls. We can also leave this step for later when we clean if we prefer. Note that the scaler will just ignore missing values, so we don't need to do this. 

In [ ]:
# Get observations where value(s) are missing
missing = pd.isnull(df["calories_scone"]) # note isnull vs isna?

# see which observations are missing
df[missing]

# let's just fill it with average
df.calories_scone = df.calories_scone.fillna(df.calories_scone.mean())

In [ ]:
# Let's use MinMaxScaler
# This require additional import statement -- added to our import section above
scaler = MinMaxScaler()

df["calories_scone_scaled"] = scaler.fit_transform(df[["calories_scone"]])

# now review the values
df["calories_scone_scaled"].describe()

In [ ]:
# Let's just see visually what this did
plt.subplot(1, 2, 1)
plt.hist(df["calories_scone"], bins=10)
plt.xlabel("calories")
plt.ylabel("frequency")

plt.subplot(1, 2, 2)
plt.hist(df["calories_scone_scaled"], bins=10)
plt.xlabel("calories")


The above shows us that the distribution of the values has not changed, only the range has changed.

Example 2: We have a feature that is not simple. We need to extract the data and create features. We don't really have a good example of this in our data above, so I've created a short example.

In [ ]:
# Our made up data
courses = ["CIT160", "CSC132", "CIT482", "EET145"]
coursesDf = pd.DataFrame(courses, columns=["Course"])
coursesDf

We want to get the subject and number in different columns. We can use other functions of the string class in Python to access these values.

Functions/functionality such as:
* str[] - this is just standard slicing
* slice()

In [ ]:
# create a new column/feature called "Subject" that contains just the subject (first 3 letters)
coursesDf["Subject"] = coursesDf["Course"].str[:3]
# use slightly different approach below for course number (slice())
coursesDf["Number"] = coursesDf["Course"].str.slice(3)
coursesDf

Let's look at some additional cleaning techniques next.

### 3. Data Cleaning

We look for dirty data (e.g. duplicates, missing data, outliers, duplicates). 

In [ ]:
# Consider the following dirty list
dirty = {
    'Name': [None, ' ', '        ', ''],
    'Age': [None, np.NaN , np.NaN, np.NaN],
    'Start': [None, pd.NaT, pd.NaT, pd.NaT],
}
dirtyDf = pd.DataFrame(dirty)
dirtyDf

As we saw last time, there are many functions what will help use here:
* duplicated() to determine which records/observations/instances are duplicates
* drop_duplicates() to drop duplicate records
* isnull() to detect missing values - returns boolean values
* isna() to detect missing values - returns boolean values (alias of above)
* dropna() to drop rows with missing values
* drop() to drop a feature/column

Will they pick up on everything "dirty" in our dirty DataFrame above?

In [ ]:
# check for missing data
dirtyDf.isna()

In [ ]:
# We can replace empty strings with a regex
dirtyDf.replace(r"^\s*$", np.nan, inplace=True, regex=True)
dirtyDf

Next we'll consider if data has outliers. There are several techniques we can use here. We can look at percentiles and remove anything outside of the 5% to 95% range.  We can use z-scores and remove anything above 3 or below -3. Standardized scores (z-scores) describe how many standard deviations—and in which direction—a value lies from the distribution's mean. 

Like addressing missing values, you should consider if these outliers are random or not, and also how they should be treated (e.g. dropped, replaced with mean/median).

In [ ]:
# Need to clean up first
# remove anything not a digit or a period - data type is wrong so there is text garbage
df["GPA"] = df["GPA"].str.replace('[^0-9\\.]', "", regex=True) # remove garbage

In [ ]:
# how many are missing
df["GPA"].isna().sum() 

In [ ]:
# convert feature to correct type and fill missing values
df["GPA"] = pd.to_numeric(df["GPA"], errors="coerce") # convert failures to NaN
df["GPA"] = df["GPA"].astype(float) #errors
df["GPA"] = df["GPA"].fillna(df["GPA"].mean())

In [ ]:
df["GPA"].isna().sum() 

In [ ]:
# find z-scores
# get mean of GPA
mean = df["GPA"].mean()
# get std of GPA
std = df["GPA"].std()
# We can add this as a new feature and then filter by it to drop rows
# zscore = (current GPA - mean)/std
df["zscore_GPA"] = (df["GPA"] - mean)/std
df["zscore_GPA"]

In [ ]:
# filter to see just outliers
# remember parens below are needed for logical &
df[(df["zscore_GPA"] > 3) | (df["zscore_GPA"] < -3)]

Note: Lab Exercise assigned